# Host Multiple Models on a Single Model Serving Endpoint

This notebook demonstrates how to package **3 distinct regression models** into a single
[MLflow PyFunc](https://mlflow.org/docs/latest/python_api/mlflow.pyfunc.html) model and
deploy them behind **one** Databricks Model Serving endpoint.

**Why consolidate?** If you run many small, traditional ML models (XGBoost, scikit-learn,
etc.) that each handle low QPS, deploying each on its own endpoint wastes compute.
A single endpoint can host all of them, routing requests by a discriminator field
(e.g. `model_name`), which dramatically reduces cost while staying well within latency SLAs.

### What this notebook covers
1. Train 3 regression models on the diabetes dataset (ElasticNet, Random Forest, Gradient Boosting)
2. Register each in **Unity Catalog**
3. Package them into one PyFunc wrapper with routing logic
4. Register the combined model in Unity Catalog
5. Create a single serving endpoint and send test requests

## 0 — Configuration
Set the catalog / schema where models will be registered.

In [0]:
CATALOG = "classic_stable_been2c_catalog"
SCHEMA = "weather"

UC_MODEL_PREFIX = f"{CATALOG}.{SCHEMA}"

MODEL_NAMES = {
    "elasticnet": f"{UC_MODEL_PREFIX}.regression_elasticnet",
    "random_forest": f"{UC_MODEL_PREFIX}.regression_random_forest",
    "gradient_boosting": f"{UC_MODEL_PREFIX}.regression_gradient_boosting",
}

MULTI_MODEL_NAME = f"{UC_MODEL_PREFIX}.multi_model_serving"
ENDPOINT_NAME = "multi-model-endpoint-demo"

## 1 — Prepare data

In [0]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn import datasets

diabetes = datasets.load_diabetes()
df = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
df["progression"] = diabetes.target

train_df, test_df = train_test_split(df, test_size=0.25, random_state=42)

train_x = train_df.drop("progression", axis=1)
train_y = train_df["progression"]
test_x = test_df.drop("progression", axis=1)
test_y = test_df["progression"]

display(train_df.head())

## 2 — Train & register 3 distinct regression models

Each model is logged with its signature and registered to Unity Catalog
with the `champion` alias.

In [0]:
import mlflow
from mlflow.models.signature import infer_signature
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

mlflow.set_registry_uri("databricks-uc")

model_configs = [
    ("elasticnet", ElasticNet(alpha=0.05, l1_ratio=0.05, random_state=42)),
    ("random_forest", RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)),
    ("gradient_boosting", GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42)),
]

for key, estimator in model_configs:
    uc_name = MODEL_NAMES[key]
    with mlflow.start_run(run_name=f"train_{key}") as run:
        estimator.fit(train_x, train_y)
        predictions = estimator.predict(test_x)
        signature = infer_signature(test_x, predictions)

        mlflow.sklearn.log_model(
            estimator,
            name="model",
            signature=signature,
            input_example=test_x.iloc[:3],
            registered_model_name=uc_name,
        )

        rmse = np.sqrt(np.mean((predictions - test_y) ** 2))
        mlflow.log_metric("rmse", rmse)
        print(f"\u2713 {key:20s}  RMSE={rmse:.2f}  \u2192 {uc_name}")

    client = mlflow.MlflowClient()
    versions = client.search_model_versions(f"name='{uc_name}'")
    latest = max(versions, key=lambda v: int(v.version))
    client.set_registered_model_alias(uc_name, "champion", latest.version)

## 3 — Build a multi-model PyFunc wrapper

The wrapper loads all 3 models at startup. Each request includes a
`model_name` field that selects which model runs inference.
This approach lets you serve N models from a single endpoint.

In [0]:
class MultiModelPyfunc(mlflow.pyfunc.PythonModel):
    """Routes inference requests to the correct sub-model based on `model_name`."""

    EXPECTED_MODELS = {"elasticnet", "random_forest", "gradient_boosting"}

    def load_context(self, context):
        self.models = {}
        for name in self.EXPECTED_MODELS:
            self.models[name] = mlflow.sklearn.load_model(context.artifacts[name])

    def predict(self, context, model_input, params=None):
        if not isinstance(model_input, pd.DataFrame):
            raise ValueError("Input must be a pandas DataFrame")

        if "model_name" not in model_input.columns:
            raise ValueError(
                "Input must contain a 'model_name' column. "
                f"Valid values: {sorted(self.EXPECTED_MODELS)}"
            )

        results = pd.Series(index=model_input.index, dtype=float)
        feature_cols = [c for c in model_input.columns if c != "model_name"]

        for name, group in model_input.groupby("model_name"):
            if name not in self.models:
                raise ValueError(
                    f"Unknown model '{name}'. "
                    f"Valid values: {sorted(self.EXPECTED_MODELS)}"
                )
            preds = self.models[name].predict(group[feature_cols])
            results.loc[group.index] = preds

        return results.values

## 4 — Log & register the combined model

In [0]:
import shutil, tempfile

staging_dir = tempfile.mkdtemp(prefix="multi_model_")
artifact_paths = {}
for key, uc_name in MODEL_NAMES.items():
    downloaded = mlflow.artifacts.download_artifacts(f"models:/{uc_name}@champion")
    dest = f"{staging_dir}/{key}"
    shutil.copytree(downloaded, dest)
    artifact_paths[key] = dest

input_example = test_x.iloc[:2].copy()
input_example["model_name"] = ["elasticnet", "random_forest"]

with mlflow.start_run(run_name="multi_model_package") as run:
    wrapped = MultiModelPyfunc()

    mlflow.pyfunc.log_model(
        name="multi_model",
        python_model=wrapped,
        artifacts=artifact_paths,
        input_example=input_example,
        registered_model_name=MULTI_MODEL_NAME,
    )
    print(f"\u2713 Combined model registered \u2192 {MULTI_MODEL_NAME}")

client = mlflow.MlflowClient()
versions = client.search_model_versions(f"name='{MULTI_MODEL_NAME}'")
latest = max(versions, key=lambda v: int(v.version))
client.set_registered_model_alias(MULTI_MODEL_NAME, "champion", latest.version)

## 5 — Local test before deploying

In [0]:
loaded = mlflow.pyfunc.load_model(f"models:/{MULTI_MODEL_NAME}@champion")

sample = test_x.iloc[:6].copy()
sample["model_name"] = ["elasticnet", "elasticnet", "random_forest",
                         "random_forest", "gradient_boosting", "gradient_boosting"]

predictions = loaded.predict(sample)

result_df = sample[["model_name"]].copy()
result_df["prediction"] = predictions
display(result_df)

## 6 — Create a Model Serving endpoint

Uses the Databricks SDK to create (or update) a single endpoint that
serves all 3 models.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)
from databricks.sdk.errors import NotFound

w = WorkspaceClient()

served_entities = [
    ServedEntityInput(
        entity_name=MULTI_MODEL_NAME,
        entity_version=latest.version,
        scale_to_zero_enabled=False,
        workload_size="Small",
    )
]

try:
    w.serving_endpoints.get(ENDPOINT_NAME)
    print(f"Endpoint '{ENDPOINT_NAME}' exists — updating served entity …")
    w.serving_endpoints.update_config_and_wait(
        name=ENDPOINT_NAME,
        served_entities=served_entities,
    )
except NotFound:
    print(f"Creating endpoint '{ENDPOINT_NAME}' …")
    w.serving_endpoints.create_and_wait(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(served_entities=served_entities),
    )

print(f"✓ Endpoint '{ENDPOINT_NAME}' is ready")

## 7a — Query a single model (the production pattern)

In production, each request targets **one** model. The `model_name` field
routes it to the right sub-model inside the PyFunc. This cell also
measures round-trip latency so you can validate against the 1 s SLA.

In [0]:
import time

sample_row = test_x.iloc[0].to_dict()

for model_name in ["elasticnet", "random_forest", "gradient_boosting"]:
    payload = [{**sample_row, "model_name": model_name}]

    start = time.time()
    response = w.serving_endpoints.query(
        name=ENDPOINT_NAME,
        dataframe_records=payload,
    )
    elapsed_ms = (time.time() - start) * 1000

    pred = response.predictions[0]
    print(f"  {model_name:25s} → {pred:>8.2f}   ({elapsed_ms:.0f} ms)")

## 7b — Batch query (multiple models in one call)

You can also send a batch of rows targeting different models in a single
HTTP request. The PyFunc groups by `model_name` and returns results
aligned with the input order.

In [0]:
batch_payload = []
for model_name in ["elasticnet", "random_forest", "gradient_boosting"]:
    batch_payload.append({**sample_row, "model_name": model_name})

start = time.time()
response = w.serving_endpoints.query(
    name=ENDPOINT_NAME,
    dataframe_records=batch_payload,
)
elapsed_ms = (time.time() - start) * 1000

for model_name, pred in zip(
    ["elasticnet", "random_forest", "gradient_boosting"],
    response.predictions,
):
    print(f"  {model_name:25s} → {pred:>8.2f}")
print(f"\n  Batch round-trip: {elapsed_ms:.0f} ms")